# PMO Benchmark Comparison

This notebook compares our LLM-based molecule optimization results against the PMO benchmark baselines.

## Data Sources
- **Our results**: `data/results/tdc_tasks/` (JSON conversation files)
- **Baseline results**: `data/results/pmo_baseline/` (YAML files from PMO benchmark)

## Metrics
We use AUC top-10 as the primary metric, following the PMO benchmark paper.

In [1]:
import json
import heapq
from pathlib import Path
import numpy as np
import pandas as pd
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

In [2]:
# Define paths
OUR_RESULTS_DIR = Path("../data/results/tdc_tasks")
PMO_RESULTS_DIR = Path("../data/results/pmo_baseline")

# Define model mappings from filename to display name (PMO baselines)
MODEL_MAPPING = {
    'reinvent': 'REINVENT',
    'reinvent_selfies': 'REINVENT SELFIES',
    'graph_ga': 'Graph GA',
    'gp_bo': 'GP BO',
    'stoned': 'STONED',
    'smiles_lstm_hc': 'SMILES LSTM HC',
    'selfies_lstm_hc': 'SELFIES LSTM HC',
    'smiles_ga': 'SMILES GA',
    'selfies_ga': 'SELFIES GA',
    'smiles_vae_bo': 'SMILES VAE BO',
    'selfies_vae_bo': 'SELFIES VAE BO',
    'jt_vae_bo': 'JT-VAE BO',
    'synnet': 'SynNet',
    'dog_gen': 'DoG-Gen',
    'dog_ae': 'DoG-AE',
    'gflownet': 'GFlowNet',
    'gflownet_al': 'GFlowNet-AL',
    'dst': 'DST',
    'mars': 'MARS',
    'mimosa': 'MIMOSA',
    'pasithea': 'Pasithea',
    'mol_pal': 'MolPAL',
    'graph_mcts': 'Graph MCTS',
    'moldqn': 'MolDQN',
    'screening': 'Screening',
}

# Our model name
OUR_MODEL_KEY = 'llm_agent'
OUR_MODEL_NAME = 'LLM Agent (Ours)'

# List of all 23 PMO benchmark tasks
TASKS = [
    'albuterol_similarity',
    'amlodipine_mpo',
    'celecoxib_rediscovery',
    'deco_hop',
    'drd2',
    'fexofenadine_mpo',
    'gsk3b',
    'isomers_c7h8n2o2',
    'isomers_c9h10n2o2pf2cl',
    'jnk3',
    'median1',
    'median2',
    'mestranol_similarity',
    'osimertinib_mpo',
    'perindopril_mpo',
    'qed',
    'ranolazine_mpo',
    'scaffold_hop',
    'sitagliptin_mpo',
    'thiothixene_rediscovery',
    'troglitazone_rediscovery',
    'valsartan_smarts',
    'zaleplon_mpo',
]

## Load Our Results

In [3]:
def load_trace(p: Path) -> list:
    """Load trace from JSON file.
    
    The trace contains a list of entries with 'iteration', 'score', and 'smiles' fields.
    """
    d = json.loads(p.read_text(encoding="utf-8"))
    return d["trace"] if isinstance(d, dict) and "trace" in d else d


def load_our_results(results_dir: Path) -> dict:
    """Load all our results from JSON trace files.
    
    Returns a dict mapping task -> list of (score, iteration) tuples for each run.
    """
    all_results = defaultdict(list)
    
    for task_dir in results_dir.iterdir():
        if not task_dir.is_dir():
            continue
        
        task_name = task_dir.name
        
        for json_file in task_dir.glob("*.json"):
            try:
                trace = load_trace(json_file)
                # Convert trace entries to (score, iteration) tuples
                results = [(float(entry["score"]), int(entry["iteration"])) for entry in trace]
                # Sort by iteration
                results.sort(key=lambda x: x[1])
                
                if results:
                    all_results[task_name].append(results)
            except Exception as e:
                print(f"Error loading {json_file}: {e}")
    
    return dict(all_results)


# Load our results
our_results = load_our_results(OUR_RESULTS_DIR)
print(f"Loaded results for {len(our_results)} tasks")
print(f"Tasks: {list(our_results.keys())}")

Loaded results for 23 tasks
Tasks: ['deco_hop', 'sitagliptin_mpo', 'troglitazone_rediscovery', 'median1', 'thiothixene_rediscovery', 'isomers_c7h8n2o2', 'valsartan_smarts', 'amlodipine_mpo', 'drd2', 'osimertinib_mpo', 'mestranol_similarity', 'celecoxib_rediscovery', 'scaffold_hop', 'albuterol_similarity', 'jnk3', 'fexofenadine_mpo', 'perindopril_mpo', 'median2', 'ranolazine_mpo', 'isomers_c9h10n2o2pf2cl', 'qed', 'gsk3b', 'zaleplon_mpo']


## Load PMO Baseline Results

In [4]:
def load_yaml_results_fast(filepath):
    """Load results from a YAML file using fast custom parsing.
    
    The YAML file format is:
    SMILES:
    - score
    - oracle_call_number
    
    Returns a list of (score, oracle_call) tuples sorted by oracle_call.
    """
    results = []
    with open(filepath, 'r') as f:
        lines = f.readlines()
    
    i = 0
    n = len(lines)
    while i < n:
        line = lines[i]
        if not line.strip():
            i += 1
            continue
        if not line.startswith('-'):
            if i + 2 < n:
                score_line = lines[i + 1].strip()
                oracle_line = lines[i + 2].strip()
                if score_line.startswith('- ') and oracle_line.startswith('- '):
                    try:
                        score = float(score_line[2:])
                        oracle_call = int(oracle_line[2:])
                        results.append((score, oracle_call))
                    except (ValueError, IndexError):
                        pass
                    i += 3
                    continue
        i += 1
    
    results.sort(key=lambda x: x[1])
    return results


# We need to load raw results and recompute AUC with k=1
# The existing cache uses k=10, so we compute fresh from YAML files

def load_pmo_raw_results():
    """Load raw PMO results from YAML files.
    
    Returns dict: model -> task -> list of [(score, oracle_call), ...] for each run
    """
    all_results = defaultdict(lambda: defaultdict(list))
    
    for model in MODEL_MAPPING.keys():
        for task in TASKS:
            pattern = f"results_{model}_{task}_*.yaml"
            for filepath in PMO_RESULTS_DIR.glob(pattern):
                results = load_yaml_results_fast(filepath)
                if results:
                    all_results[model][task].append(results)
    
    return all_results


print("Loading PMO baseline raw results from YAML files...")
pmo_raw_results = load_pmo_raw_results()
n_files = sum(len(runs) for model_data in pmo_raw_results.values() for runs in model_data.values())
print(f"Loaded {n_files} result files for {len(pmo_raw_results)} models")

Loading PMO baseline raw results from YAML files...
Loaded 2875 result files for 25 models


## Compute AUC Top-1 for Our Results

In [5]:
MAX_ORACLE_CALLS = 10000  # Standard PMO benchmark limit


def compute_auc_top_1(results, max_oracle_calls=10000):
    """Compute the AUC of top-1 (best) score vs oracle calls.
    
    Args:
        results: List of (score, oracle_call) tuples sorted by oracle_call
        max_oracle_calls: Maximum number of oracle calls (default 10000)
    
    Returns:
        AUC value normalized to [0, 1]
    """
    if not results:
        return 0.0
    
    best_score = 0.0
    best_at_call = {}
    
    for score, oracle_call in results:
        if oracle_call > max_oracle_calls:
            break
        
        if score > best_score:
            best_score = score
        
        best_at_call[oracle_call] = best_score
    
    if not best_at_call:
        return 0.0
    
    oracle_calls = sorted(best_at_call.keys())
    
    # Compute AUC as sum of rectangles
    auc = 0.0
    prev_call = 0
    prev_best = 0.0
    
    for call in oracle_calls:
        auc += prev_best * (call - prev_call)
        prev_call = call
        prev_best = best_at_call[call]
    
    # Add final rectangle from last call to max_oracle_calls
    # This extends the best value to max_oracle_calls
    auc += prev_best * (max_oracle_calls - prev_call)
    
    # Normalize by max possible AUC
    normalized_auc = auc / max_oracle_calls
    
    return normalized_auc


# Compute AUC Top-1 for our results
# Our experiments use 51 iterations, but we extend to 10000 by holding best value constant
our_auc_results = defaultdict(list)

for task, runs in our_results.items():
    for run_results in runs:
        # Compute AUC over 10000 oracle calls
        # The best value at iteration 51 will be held constant until 10000
        auc = compute_auc_top_1(run_results, max_oracle_calls=MAX_ORACLE_CALLS)
        our_auc_results[task].append(auc)

print("AUC Top-1 for our results (extended to 10000 oracle calls):")
for task, aucs in sorted(our_auc_results.items()):
    print(f"  {task}: {np.mean(aucs):.4f} (n={len(aucs)})")

AUC Top-1 for our results (extended to 10000 oracle calls):
  albuterol_similarity: 0.5973 (n=1)
  amlodipine_mpo: 0.9055 (n=1)
  celecoxib_rediscovery: 0.3496 (n=1)
  deco_hop: 0.9649 (n=1)
  drd2: 0.9999 (n=1)
  fexofenadine_mpo: 0.9899 (n=1)
  gsk3b: 0.9999 (n=1)
  isomers_c7h8n2o2: 0.9999 (n=1)
  isomers_c9h10n2o2pf2cl: 0.9999 (n=1)
  jnk3: 0.4998 (n=1)
  median1: 0.3999 (n=1)
  median2: 0.3623 (n=1)
  mestranol_similarity: 0.4126 (n=1)
  osimertinib_mpo: 0.9119 (n=1)
  perindopril_mpo: 0.8103 (n=1)
  qed: 0.9446 (n=1)
  ranolazine_mpo: 0.9403 (n=1)
  scaffold_hop: 0.9998 (n=1)
  sitagliptin_mpo: 0.3383 (n=1)
  thiothixene_rediscovery: 0.7351 (n=1)
  troglitazone_rediscovery: 0.2449 (n=1)
  valsartan_smarts: 0.0000 (n=1)
  zaleplon_mpo: 0.7295 (n=1)


In [6]:
# Compute AUC Top-1 for PMO baselines
pmo_auc_results = defaultdict(lambda: defaultdict(list))

for model, task_data in pmo_raw_results.items():
    for task, runs in task_data.items():
        for run_results in runs:
            auc = compute_auc_top_1(run_results, max_oracle_calls=MAX_ORACLE_CALLS)
            pmo_auc_results[model][task].append(auc)

print(f"Computed AUC Top-1 for {len(pmo_auc_results)} PMO baseline models")

Computed AUC Top-1 for 25 PMO baseline models


## Create Comparison Table

In [7]:
def create_comparison_table(pmo_results_dict, our_results_dict, models_order):
    """Create a comparison table with mean ± std for each model/task combination."""
    rows = []
    
    for task in TASKS:
        row = {'Task': task}
        
        # Add our results first
        if task in our_results_dict:
            values = our_results_dict[task]
            mean = np.mean(values)
            if len(values) > 1:
                std = np.std(values)
                row[OUR_MODEL_NAME] = f"{mean:.3f}± {std:.3f}"
            else:
                row[OUR_MODEL_NAME] = f"{mean:.3f}"
        else:
            row[OUR_MODEL_NAME] = "-"
        
        # Add PMO baselines
        for model in models_order:
            if model in pmo_results_dict and task in pmo_results_dict[model]:
                values = pmo_results_dict[model][task]
                mean = np.mean(values)
                std = np.std(values)
                row[MODEL_MAPPING[model]] = f"{mean:.3f}± {std:.3f}"
            else:
                row[MODEL_MAPPING[model]] = "-"
        rows.append(row)
    
    # Add sum row
    sum_row = {'Task': 'Sum'}
    
    # Our sum
    our_total = 0
    for task in TASKS:
        if task in our_results_dict:
            our_total += np.mean(our_results_dict[task])
    sum_row[OUR_MODEL_NAME] = f"{our_total:.3f}"
    
    # PMO baselines sums
    for model in models_order:
        if model in pmo_results_dict:
            total = 0
            for task in TASKS:
                if task in pmo_results_dict[model]:
                    total += np.mean(pmo_results_dict[model][task])
            sum_row[MODEL_MAPPING[model]] = f"{total:.3f}"
        else:
            sum_row[MODEL_MAPPING[model]] = "-"
    rows.append(sum_row)
    
    df = pd.DataFrame(rows)
    return df

In [8]:
# Calculate sum for each PMO model
all_pmo_models = list(pmo_auc_results.keys())
model_sums = {}
for model in all_pmo_models:
    total = sum(np.mean(pmo_auc_results[model][task]) for task in TASKS if task in pmo_auc_results[model])
    model_sums[model] = total

# Sort models by sum (descending)
sorted_pmo_models = sorted(all_pmo_models, key=lambda m: model_sums[m], reverse=True)

print("\n" + "="*100)
print("COMPARISON TABLE: LLM Agent vs PMO Baselines (AUC Top-1)")
print("="*100 + "\n")

comparison_table = create_comparison_table(pmo_auc_results, our_auc_results, sorted_pmo_models)

# Add rank row
# Calculate our rank based on sum
our_sum = sum(np.mean(our_auc_results[task]) for task in TASKS if task in our_auc_results)
all_sums = [(OUR_MODEL_KEY, our_sum)] + [(m, model_sums[m]) for m in sorted_pmo_models]
all_sums_sorted = sorted(all_sums, key=lambda x: x[1], reverse=True)
rank_map = {model: i+1 for i, (model, _) in enumerate(all_sums_sorted)}

rank_row = {'Task': 'Rank'}
rank_row[OUR_MODEL_NAME] = str(rank_map[OUR_MODEL_KEY])
for model in sorted_pmo_models:
    rank_row[MODEL_MAPPING[model]] = str(rank_map[model])

comparison_table = pd.concat([comparison_table, pd.DataFrame([rank_row])], ignore_index=True)

display(comparison_table)


COMPARISON TABLE: LLM Agent vs PMO Baselines (AUC Top-1)



,Task,LLM Agent (Ours),REINVENT,Graph GA,REINVENT SELFIES,GP BO,SMILES LSTM HC,STONED,DoG-Gen,SynNet,SMILES GA,MolPAL,DST,MARS,SELFIES LSTM HC,MIMOSA,DoG-AE,SELFIES VAE BO,Screening,SMILES VAE BO,Pasithea,GFlowNet,JT-VAE BO,GFlowNet-AL,SELFIES GA,Graph MCTS,MolDQN
0,albuterol_similarity,0.597,0.905± 0.003,0.877± 0.022,0.855± 0.033,0.925± 0.011,0.800± 0.031,0.756± 0.078,0.749± 0.014,0.647± 0.052,0.680± 0.056,0.697± 0.003,0.673± 0.022,0.671± 0.121,0.728± 0.029,0.651± 0.024,0.623± 0.045,0.574± 0.044,0.549± 0.029,0.565± 0.020,0.501± 0.005,0.503± 0.030,0.543± 0.051,0.442± 0.020,0.529± 0.030,0.627± 0.028,0.349± 0.023
1,amlodipine_mpo,0.906,0.655± 0.037,0.688± 0.021,0.628± 0.021,0.609± 0.044,0.638± 0.021,0.618± 0.048,0.557± 0.005,0.583± 0.006,0.567± 0.004,0.623± 0.011,0.576± 0.048,0.525± 0.023,0.571± 0.007,0.593± 0.009,0.536± 0.013,0.582± 0.005,0.582± 0.015,0.604± 0.032,0.585± 0.000,0.469± 0.007,0.585± 0.000,0.450± 0.007,0.421± 0.033,0.473± 0.020,0.344± 0.013
2,celecoxib_rediscovery,0.350,0.803± 0.098,0.684± 0.123,0.618± 0.039,0.809± 0.075,0.621± 0.030,0.389± 0.044,0.526± 0.013,0.487± 0.032,0.352± 0.027,0.498± 0.002,0.424± 0.005,0.429± 0.049,0.427± 0.015,0.422± 0.017,0.403± 0.025,0.388± 0.023,0.396± 0.005,0.409± 0.013,0.353± 0.011,0.375± 0.008,0.387± 0.026,0.290± 0.005,0.242± 0.023,0.297± 0.009,0.115± 0.017
3,deco_hop,0.965,0.682± 0.048,0.627± 0.005,0.649± 0.022,0.648± 0.027,0.891± 0.009,0.616± 0.009,0.877± 0.004,0.630± 0.012,0.617± 0.007,0.807± 0.019,0.622± 0.011,0.601± 0.003,0.605± 0.004,0.629± 0.005,0.844± 0.010,0.593± 0.002,0.614± 0.003,0.611± 0.004,0.606± 0.012,0.594± 0.002,0.598± 0.004,0.594± 0.004,0.556± 0.006,0.564± 0.003,0.552± 0.002
4,drd2,1.000,0.968± 0.007,0.990± 0.001,0.979± 0.003,0.958± 0.007,0.957± 0.012,0.934± 0.020,0.995± 0.000,0.985± 0.003,0.931± 0.017,0.903± 0.007,0.887± 0.021,0.939± 0.015,0.848± 0.038,0.880± 0.023,0.986± 0.003,0.808± 0.055,0.798± 0.060,0.820± 0.074,0.559± 0.087,0.792± 0.041,0.742± 0.185,0.717± 0.074,0.426± 0.208,0.477± 0.111,0.030± 0.004
5,fexofenadine_mpo,0.990,0.804± 0.007,0.777± 0.011,0.765± 0.005,0.743± 0.007,0.756± 0.011,0.806± 0.018,0.733± 0.008,0.782± 0.018,0.733± 0.017,0.706± 0.001,0.745± 0.005,0.732± 0.008,0.719± 0.006,0.724± 0.013,0.719± 0.042,0.701± 0.006,0.693± 0.012,0.702± 0.009,0.705± 0.040,0.717± 0.007,0.698± 0.013,0.717± 0.005,0.608± 0.008,0.597± 0.011,0.500± 0.016
6,gsk3b,1.000,0.893± 0.045,0.829± 0.070,0.824± 0.035,0.879± 0.040,0.938± 0.015,0.704± 0.055,0.960± 0.008,0.855± 0.044,0.669± 0.039,0.776± 0.002,0.737± 0.037,0.630± 0.056,0.539± 0.040,0.641± 0.046,0.756± 0.118,0.507± 0.091,0.658± 0.079,0.537± 0.047,0.402± 0.076,0.691± 0.033,0.482± 0.055,0.641± 0.030,0.363± 0.023,0.355± 0.032,0.287± 0.012
7,isomers_c7h8n2o2,1.000,0.884± 0.029,0.899± 0.060,0.890± 0.034,0.747± 0.112,0.616± 0.060,0.914± 0.010,0.580± 0.034,0.608± 0.050,0.931± 0.021,0.832± 0.005,0.665± 0.075,0.807± 0.048,0.696± 0.025,0.636± 0.057,0.550± 0.188,0.498± 0.053,0.396± 0.079,0.332± 0.052,0.793± 0.057,0.540± 0.069,0.244± 0.075,0.450± 0.097,0.879± 0.013,0.702± 0.048,0.595± 0.078
8,isomers_c9h10n2o2pf2cl,1.000,0.673± 0.059,0.766± 0.047,0.781± 0.024,0.514± 0.172,0.466± 0.034,0.823± 0.029,0.365± 0.031,0.434± 0.085,0.881± 0.062,0.361± 0.009,0.551± 0.040,0.640± 0.023,0.476± 0.038,0.346± 0.045,0.134± 0.072,0.367± 0.083,0.218± 0.048,0.176± 0.032,0.499± 0.081,0.174± 0.046,0.274± 0.122,0.131± 0.024,0.682± 0.023,0.601± 0.067,0.481± 0.043
9,jnk3,0.500,0.814± 0.025,0.598± 0.142,0.671± 0.069,0.593± 0.160,0.788± 0.057,0.543± 0.094,0.709± 0.022,0.723± 0.042,0.340± 0.025,0.458± 0.025,0.589± 0.069,0.549± 0.089,0.304± 0.053,0.403± 0.072,0.540± 0.133,0.342± 0.070,0.363± 0.064,0.376± 0.054,0.207± 0.033,0.480± 0.026,0.354± 0.064,0.429± 0.038,0.235± 0.022,0.145± 0.032,0.135± 0.014


## Summary Statistics

In [9]:
# Print summary
print("\n" + "="*60)
print("SUMMARY")
print("="*60)

# Tasks we have results for
our_tasks = set(our_auc_results.keys())
pmo_tasks = set(TASKS)
common_tasks = our_tasks & pmo_tasks
missing_tasks = pmo_tasks - our_tasks
extra_tasks = our_tasks - pmo_tasks

print(f"\nTasks in PMO benchmark: {len(pmo_tasks)}")
print(f"Tasks with our results: {len(our_tasks)}")
print(f"Common tasks: {len(common_tasks)}")

if missing_tasks:
    print(f"\nMissing tasks (in PMO but not in ours): {sorted(missing_tasks)}")
if extra_tasks:
    print(f"\nExtra tasks (in ours but not in PMO): {sorted(extra_tasks)}")

# Our rank
print(f"\n" + "-"*40)
print(f"Our rank: {rank_map[OUR_MODEL_KEY]} out of {len(all_sums)}")
print(f"Our sum: {our_sum:.3f}")
print(f"Top model sum: {all_sums_sorted[0][1]:.3f} ({MODEL_MAPPING.get(all_sums_sorted[0][0], all_sums_sorted[0][0])})")


SUMMARY

Tasks in PMO benchmark: 23
Tasks with our results: 23
Common tasks: 23

----------------------------------------
Our rank: 1 out of 26
Our sum: 16.136
Top model sum: 16.136 (llm_agent)


In [10]:
# Show top 10 models by sum
print("\nTop 10 models by sum:")
print("-" * 40)
for i, (model, total) in enumerate(all_sums_sorted[:10], 1):
    name = OUR_MODEL_NAME if model == OUR_MODEL_KEY else MODEL_MAPPING.get(model, model)
    marker = " <-- Ours" if model == OUR_MODEL_KEY else ""
    print(f"{i:2d}. {name:25s} {total:.3f}{marker}")


Top 10 models by sum:
----------------------------------------
 1. LLM Agent (Ours)          16.136 <-- Ours
 2. REINVENT                  14.742
 3. Graph GA                  14.383
 4. REINVENT SELFIES          14.103
 5. GP BO                     13.827
 6. SMILES LSTM HC            13.643
 7. STONED                    13.288
 8. DoG-Gen                   12.752
 9. SynNet                    12.459
10. SMILES GA                 12.388
